<a href="https://colab.research.google.com/github/bhanu613/alias-aware-technical-skill-extraction/blob/main/notebooks/02%20System%20Implementation%20and%20Testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluating Alias-Aware Matching for Technical-Skill Extraction

## System Implementation and Testing

This notebook implements the two frozen lexicon-based extractors used in the
held-out evaluation.

System A searches the 20 canonical technical-skill forms. System B uses the
same matching policy and adds 13 reviewed aliases, while returning the same
canonical output labels.

The notebook loads the frozen lexicon and test cases from the repository,
defines the matching functions, and verifies them with 41 safety tests and
6 multi-skill integration tests. The lexicon, aliases, normalisation policy,
and boundary rules are not changed here.

In [1]:
from pathlib import Path

import os
import subprocess
import sys
import json

import pandas as pd


repositoryUrl = (
    "https://github.com/bhanu613/"
    "alias-aware-technical-skill-extraction.git"
)

repositoryFolder = Path(
    "/content/alias-aware-technical-skill-extraction"
)


if not repositoryFolder.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            repositoryUrl,
            str(repositoryFolder)
        ],
        check=True
    )


os.chdir(repositoryFolder)


requirementsPath = repositoryFolder / "requirements.txt"

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(requirementsPath)
    ],
    check=True
)


dataFolder = repositoryFolder / "data"

configFolder = repositoryFolder / "config"

runtimeFolder = Path("/content/evaluationOutputs")

runtimeFolder.mkdir(
    parents=True,
    exist_ok=True
)


lexiconPath = configFolder / "LexiconList20_final.json"

safetyTestsPath = dataFolder / "SafetyTestCases.csv"

integrationTestsPath = (
    dataFolder / "IntegrationTestCases.csv"
)


with lexiconPath.open(
    "r",
    encoding="utf-8"
) as file:
    finalLexicon = json.load(file)


safetyTests = pd.read_csv(
    safetyTestsPath
)

integrationTests = pd.read_csv(
    integrationTestsPath
)


skills = finalLexicon["skills"]

canonicalLabels = set(
    skills.keys()
)

aliasCount = sum(
    len(details["aliases"])
    for details in skills.values()
)


assert len(canonicalLabels) == 20, (
    f"Expected 20 canonical labels, "
    f"found {len(canonicalLabels)}."
)

assert aliasCount == 13, (
    f"Expected 13 approved aliases, "
    f"found {aliasCount}."
)

assert len(safetyTests) == 41, (
    f"Expected 41 safety tests, "
    f"found {len(safetyTests)}."
)

assert len(integrationTests) == 6, (
    f"Expected 6 integration-test cases, "
    f"found {len(integrationTests)}."
)


print("Frozen implementation inputs loaded successfully.")
print(f"Canonical labels: {len(canonicalLabels)}")
print(f"Approved aliases: {aliasCount}")
print(f"Safety-test cases: {len(safetyTests)}")
print(f"Integration-test records: {len(integrationTests)}")
print(f"Repository folder: {repositoryFolder}")

Frozen implementation inputs loaded successfully.
Canonical labels: 20
Approved aliases: 13
Safety-test cases: 41
Integration-test records: 6
Repository folder: /content/alias-aware-technical-skill-extraction


## Shared Matcher

The frozen matching functions are stored in `matcher.py` at the repository
root. Keeping one authoritative matcher ensures that the system tested in this
notebook is the same system used later for held-out prediction and evaluation.

The matcher normalises text, creates boundary-safe patterns, searches canonical
forms and approved aliases, and always returns canonical output labels.

## Safety and Integration Checks

The frozen matcher is tested before it is applied to held-out evaluation
documents.

The safety suite checks individual canonical forms, approved aliases, substring
protection, boundary behaviour, and documented hyphen-policy outcomes.

The integration suite checks complete multi-skill outputs, including mixed
canonical and alias forms, duplicate forms, and negative strings.

In [2]:
from matcher import (
    extractSystemA,
    extractSystemB,
    loadLexicon,
    normaliseText,
    termPattern,
    validateLexicon
)


finalLexicon = loadLexicon(
    lexiconPath
)

skills, canonicalLabels, aliasCount = validateLexicon(
    finalLexicon
)


print("Shared matcher loaded successfully.")
print(f"Canonical labels: {len(canonicalLabels)}")
print(f"Approved aliases: {aliasCount}")

Shared matcher loaded successfully.
Canonical labels: 20
Approved aliases: 13


In [3]:
requiredSafetyColumns = {
    "text",
    "Canonical label",
    "Expected System A",
    "Expected System B",
    "Reason"
}


assert requiredSafetyColumns.issubset(
    safetyTests.columns
), (
    "SafetyTestCases.csv is missing one or more required columns."
)


safetyResults = safetyTests.copy()


safetyResults["System A predictions"] = (
    safetyResults["text"].apply(
        lambda text: extractSystemA(
            text,
            skills
        )
    )
)

safetyResults["System B predictions"] = (
    safetyResults["text"].apply(
        lambda text: extractSystemB(
            text,
            skills
        )
    )
)


safetyResults["System A observed"] = (
    safetyResults.apply(
        lambda row: (
            row["Canonical label"]
            in row["System A predictions"]
        ),
        axis=1
    )
)

safetyResults["System B observed"] = (
    safetyResults.apply(
        lambda row: (
            row["Canonical label"]
            in row["System B predictions"]
        ),
        axis=1
    )
)


safetyResults["System A passes"] = (
    safetyResults["System A observed"]
    == safetyResults["Expected System A"]
)

safetyResults["System B passes"] = (
    safetyResults["System B observed"]
    == safetyResults["Expected System B"]
)

safetyResults["Test passes"] = (
    safetyResults["System A passes"]
    & safetyResults["System B passes"]
)


numberPassed = int(
    safetyResults["Test passes"].sum()
)

numberFailed = int(
    len(safetyResults)
    - numberPassed
)


safetySummary = pd.DataFrame(
    {
        "Safety-test cases": [
            len(safetyResults)
        ],
        "Passed": [
            numberPassed
        ],
        "Failed": [
            numberFailed
        ]
    }
)


display(safetySummary)


safetyFailures = safetyResults.loc[
    ~safetyResults["Test passes"],
    [
        "text",
        "Canonical label",
        "Expected System A",
        "System A observed",
        "Expected System B",
        "System B observed",
        "Reason"
    ]
].copy()


if len(safetyFailures) == 0:
    print("All 41 safety tests passed.")
else:
    print("Safety-test failures require implementation review.")
    display(safetyFailures)


assert numberPassed == 41, (
    f"Expected 41 passing safety tests, "
    f"found {numberPassed}."
)

assert numberFailed == 0, (
    f"Expected 0 failed safety tests, "
    f"found {numberFailed}."
)


safetyResults.to_csv(
    runtimeFolder / "SafetyTestResults.csv",
    index=False
)

,Safety-test cases,Passed,Failed
0,41,41,0


All 41 safety tests passed.


In [4]:
import ast


requiredIntegrationColumns = {
    "Test name",
    "text",
    "Expected System A predictions",
    "Expected System B predictions",
    "Reason"
}


assert requiredIntegrationColumns.issubset(
    integrationTests.columns
), (
    "IntegrationTestCases.csv is missing one or more "
    "required columns."
)


def parseExpectedLabels(value):

    if pd.isna(value):
        return set()

    text = str(value).strip()

    if text in {"", "[]", "set()"}:
        return set()

    parsedValue = ast.literal_eval(
        text
    )

    return set(
        parsedValue
    )


integrationChecks = integrationTests.copy()


integrationChecks["Expected System A labels"] = (
    integrationChecks["Expected System A predictions"].apply(
        parseExpectedLabels
    )
)

integrationChecks["Expected System B labels"] = (
    integrationChecks["Expected System B predictions"].apply(
        parseExpectedLabels
    )
)


integrationChecks["Observed System A labels"] = (
    integrationChecks["text"].apply(
        lambda text: set(
            extractSystemA(
                text,
                skills
            )
        )
    )
)

integrationChecks["Observed System B labels"] = (
    integrationChecks["text"].apply(
        lambda text: set(
            extractSystemB(
                text,
                skills
            )
        )
    )
)


integrationChecks["System A passes"] = (
    integrationChecks["Observed System A labels"]
    == integrationChecks["Expected System A labels"]
)

integrationChecks["System B passes"] = (
    integrationChecks["Observed System B labels"]
    == integrationChecks["Expected System B labels"]
)

integrationChecks["Test passes"] = (
    integrationChecks["System A passes"]
    & integrationChecks["System B passes"]
)


integrationPassed = int(
    integrationChecks["Test passes"].sum()
)

integrationFailed = int(
    len(integrationChecks)
    - integrationPassed
)


integrationSummary = pd.DataFrame(
    {
        "Integration-test cases": [
            len(integrationChecks)
        ],
        "Passed": [
            integrationPassed
        ],
        "Failed": [
            integrationFailed
        ]
    }
)


display(integrationSummary)


integrationFailures = integrationChecks.loc[
    ~integrationChecks["Test passes"],
    [
        "Test name",
        "text",
        "Expected System A labels",
        "Observed System A labels",
        "Expected System B labels",
        "Observed System B labels",
        "Reason"
    ]
].copy()


if len(integrationFailures) == 0:
    print("All 6 integration tests passed.")
else:
    print("Integration-test failures require implementation review.")
    display(integrationFailures)


assert integrationPassed == 6, (
    f"Expected 6 passing integration tests, "
    f"found {integrationPassed}."
)

assert integrationFailed == 0, (
    f"Expected 0 failed integration tests, "
    f"found {integrationFailed}."
)


integrationChecks.to_csv(
    runtimeFolder / "IntegrationTestResults.csv",
    index=False
)

,Integration-test cases,Passed,Failed
0,6,6,0


All 6 integration tests passed.


## Test Interpretation

A passing safety or integration test confirms that the implemented matcher follows
the frozen lexicon, normalisation, alias, and boundary policy on the specified
test cases.

The tests do not decide whether an alias should be accepted. Alias decisions
were made earlier from development-only coverage evidence, contextual review,
technical equivalence, extraction value, ambiguity, and redundancy.

The held-out evaluation in Notebook 4 measures performance on independently
annotated job descriptions.

## Implementation Status

The shared matcher is used by both System A and System B.

System A searches canonical forms only.

System B searches the same canonical forms and adds the frozen reviewed aliases.

A successful execution of this notebook confirms that the matcher passes all
41 safety cases and all 6 multi-skill integration cases. The same matcher is
then used without modification in the held-out evaluation.